In [1]:
from pathlib import Path
import random

candidates = [
    Path("dalg-cache/pile_gemma2b_100M_windows"),
    Path("../dalg-cache/pile_gemma2b_100M_windows"),
]
root = next(p.resolve() for p in candidates if p.exists())
ds_path = root / "merged"

print(ds_path)
print("exists:", ds_path.exists())
sorted(p.name for p in ds_path.iterdir())[:5], len(list(ds_path.glob("*.arrow")))

/orfeo/cephfs/scratch/dssc/zenocosini/dalg-cache/pile_gemma2b_100M_windows/merged
exists: True


(['data-00000-of-00020.arrow',
  'data-00001-of-00020.arrow',
  'data-00002-of-00020.arrow',
  'data-00003-of-00020.arrow',
  'data-00004-of-00020.arrow'],
 20)

In [2]:
from datasets import load_from_disk

ds = load_from_disk(str(ds_path))
ds

/u/dssc/zenocosini/decomposing-activations-local-geometry/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['text', 'subset', 'token_ids', 'window_start', 'window_end', 'doc_len'],
    num_rows: 328964
})

In [6]:
sorted(set(ds["subset"]))

['pile-arxiv',
 'pile-dm_mathematics',
 'pile-enron_emails',
 'pile-europarl',
 'pile-freelaw',
 'pile-github',
 'pile-gutenberg_pg-19',
 'pile-hackernews',
 'pile-nih_exporter',
 'pile-philpapers',
 'pile-pile-cc',
 'pile-pubmed_abstracts',
 'pile-pubmed_central',
 'pile-stackexchange',
 'pile-ubuntu_irc-broken',
 'pile-uspto_backgrounds',
 'pile-wikipedia_en']

In [7]:
ds_sub = ds.filter(lambda row: row["subset"] == "pile-wikipedia_en")
len(ds_sub)

Filter: 100%|██████████| 328964/328964 [01:16<00:00, 4301.77 examples/s] 


17299

In [9]:
ds_sub[0]

{'text': "Major League Baseball All-Century Team\n\nIn 1999, the Major League Baseball All-Century Team was chosen by popular vote of fans. To select the team, a panel of experts first compiled a list of the 100 greatest Major League Baseball players from the past century. Over two million fans then voted on the players using paper and online ballots.\n\nThe top two vote-getters from each position, except outfielders (nine), and the top six pitchers were placed on the team. A select panel then added five legends to create a thirty-man team:—Warren Spahn (who finished #10 among pitchers), Christy Mathewson (#14 among pitchers), Lefty Grove (#18 among pitchers), Honus Wagner (#4 among shortstops), and Stan Musial (#11 among outfielders).\n\nThe nominees for the All-Century team were presented at the 1999 All-Star Game at Fenway Park. Preceding Game 2 of the 1999 World Series, the members of the All-Century Team were revealed. Every living player named to the team attended.\n\nFor the com

In [ ]:
print("rows:", len(ds))
print("columns:", ds.column_names)
print("features:", ds.features)

for col in ["subset", "window_start", "window_end", "doc_len"]:
    print(col, ds[0][col])

In [ ]:
i = 0
row = ds[i]

print("row", i)
print("subset:", row["subset"])
print("window:", row["window_start"], row["window_end"], "doc_len:", row["doc_len"])
print("num tokens:", len(row["token_ids"]))
print(row["text"][:2000])

In [ ]:
for i in random.sample(range(len(ds)), 5):
    row = ds[i]
    preview = row["text"].replace("\n", " ")[:240]
    print(f"[{i}] subset={row['subset']} window={row['window_start']}:{row['window_end']} doc_len={row['doc_len']}")
    print(preview)
    print("-" * 100)

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("google/gemma-2b")

def show(i, start=0, end=None):
    row = ds[int(i)]
    ids = row["token_ids"][start:end]
    print(f"row={i} subset={row['subset']} window={row['window_start']}:{row['window_end']} tokens={len(row['token_ids'])}")
    print(tok.decode(ids))

show(0)

In [ ]:
def show_token_context(i, pos, pad=20):
    row = ds[int(i)]
    ids = row["token_ids"]
    lo = max(0, pos - pad)
    hi = min(len(ids), pos + pad + 1)
    left = tok.decode(ids[lo:pos])
    center = tok.decode(ids[pos:pos + 1])
    right = tok.decode(ids[pos + 1:hi])
    print(f"row={i} token_pos={pos} global_doc_token={row['window_start'] + pos}")
    print(left + " [" + center + "] " + right)

show_token_context(0, 32)

In [ ]:
def search_text(needle, limit=10, scan_rows=50_000):
    needle_l = needle.lower()
    hits = []
    for i, row in enumerate(ds.select(range(min(scan_rows, len(ds))))):
        if needle_l in row["text"].lower():
            hits.append(i)
            print(f"[{i}] subset={row['subset']} window={row['window_start']}:{row['window_end']}")
            print(row["text"].replace("\n", " ")[:500])
            print("-" * 100)
            if len(hits) >= limit:
                break
    return hits

hits = search_text("neuron", limit=3)

In [ ]:
from collections import Counter

n = min(100_000, len(ds))
Counter(ds.select(range(n))["subset"]).most_common()